# Chain of Thought RAG with AWS

## 📖 Overview

This notebook demonstrates **Chain of Thought RAG** using AWS services:
- **Step-by-step reasoning** broken into explicit steps
- **Retrieval at each step** for targeted information
- **Intermediate reasoning** visible and traceable
- **Progressive answer building** through logical flow

### What is Chain of Thought RAG?

**Traditional RAG (Single-Shot):**
```
Query → Retrieve All → Generate Complete Answer
│
└─ Black box! No visibility into reasoning process
```

**Chain of Thought RAG:**
```
Query → Break into Steps
         ↓
         Step 1: Retrieve → Reason → Intermediate Result
         ↓
         Step 2: Retrieve → Reason → Build on Step 1
         ↓
         Step 3: Retrieve → Reason → Final Answer
```

### Key Concepts

1. **Reasoning Decomposition**: Break complex query into steps
   - "First, I need to understand X..."
   - "Then, I need to compare Y..."
   - "Finally, I can conclude Z..."

2. **Step-wise Retrieval**: Get information as needed
   - Not all at once upfront
   - Targeted queries per step
   - Build knowledge progressively

3. **Intermediate Reasoning**: Think through each step
   - Visible thought process
   - Traceable logic flow
   - Debug-friendly

4. **Progressive Building**: Each step informs the next
   - Step 2 uses result from Step 1
   - Step 3 uses results from Steps 1 & 2
   - Cumulative knowledge

### Why Chain of Thought?

**Problems it Solves:**
- ❌ Black-box reasoning is hard to debug
- ❌ Complex queries need logical steps
- ❌ Single retrieval may miss needed info
- ❌ Can't verify reasoning process
- ❌ Difficult to explain how answer was reached

**Chain of Thought Solutions:**
- ✅ Transparent step-by-step reasoning
- ✅ Break complexity into manageable pieces
- ✅ Retrieve information as needed
- ✅ Traceable and verifiable logic
- ✅ Explainable answer generation

### When to Use

✅ **Good for:**
- Complex multi-step reasoning
- Math or logic problems
- Need explainable reasoning
- Debugging poor answers
- Educational applications
- Analysis requiring multiple stages

❌ **Not ideal for:**
- Simple factual Q&A
- When reasoning steps aren't needed
- Tight latency requirements
- Very simple queries
- Casual conversational use

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Decompose into<br/>Reasoning Steps]
    
    B --> C1[Step 1:<br/>Understand Problem]
    
    C1 --> D1[Retrieve<br/>Relevant Concepts]
    D1 --> E1[Reason:<br/>What do we know?]
    E1 --> F1[Intermediate Result 1]
    
    F1 --> C2[Step 2:<br/>Gather Evidence]
    
    C2 --> D2[Retrieve<br/>Specific Facts]
    D2 --> E2[Reason:<br/>How does this relate?]
    E2 --> F2[Intermediate Result 2]
    
    F2 --> C3[Step 3:<br/>Draw Conclusion]
    
    C3 --> D3[Retrieve<br/>Supporting Data]
    D3 --> E3[Reason:<br/>What's the answer?]
    E3 --> F3[Final Answer]
    
    F1 -.Context.-> C2
    F2 -.Context.-> C3
    
    F3 --> G[Return Answer<br/>+ Reasoning Chain]
    
    style A fill:#e1f5ff
    style B fill:#fff3e0
    style E1 fill:#fff9c4
    style E2 fill:#fff9c4
    style E3 fill:#fff9c4
    style F3 fill:#c8e6c9
    style G fill:#b3e5fc
```

## 1️⃣ Setup & Imports

In [1]:
import sys
import json
from typing import List, Dict, Optional
import time
from dataclasses import dataclass

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

# Expected output:
# ✓ Imports successful

✓ Imports successful


## 2️⃣ Configuration

In [2]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'chain_of_thought_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-sonnet-4-6'  # Need Opus for reasoning

# Chain of Thought Parameters
MAX_REASONING_STEPS = 5  # Maximum steps in reasoning chain
RETRIEVAL_TOP_K = 3  # Docs per step (smaller than single-shot)

print(f"Configuration:")
print(f"  Model: {LLM_MODEL.split('.')[-1][:40]}")
print(f"  Max reasoning steps: {MAX_REASONING_STEPS}")
print(f"  Retrieval per step: Top-{RETRIEVAL_TOP_K}")

# Expected output:
# Configuration:
#   Model: claude-opus-4-1-20250805-v1:0
#   Max reasoning steps: 5
#   Retrieval per step: Top-3

Configuration:
  Model: claude-sonnet-4-6
  Max reasoning steps: 5
  Retrieval per step: Top-3


## 3️⃣ Initialize Services

In [3]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

# Expected output:
# ✓ Services initialized

✓ Services initialized


## 4️⃣ Load Knowledge Base

In [4]:
sample_documents = [
    # AWS Bedrock pricing
    "Claude Opus on AWS Bedrock: $15 per million input tokens, $75 per million output tokens.",
    "Claude Sonnet on Bedrock: $3 per million input tokens, $15 per million output tokens.",
    "Claude Haiku on Bedrock: $0.25 per million input tokens, $1.25 per million output tokens.",
    "Bedrock batch inference offers 50% discount compared to real-time inference.",
    
    # Token calculations
    "Approximate tokens: 1 token ≈ 4 characters or 0.75 words in English.",
    "1,000 tokens ≈ 750 words or 3-4 paragraphs of text.",
    "Average RAG query uses 500 input tokens (context) + 200 output tokens (answer).",
    
    # RAG costs
    "Simple RAG with Opus: ~$0.05 per query (context + generation).",
    "Embeddings cost: Titan V2 at $0.0001 per 1K tokens, negligible in RAG.",
    "OpenSearch vector search: compute-based pricing, typically $0.001 per query.",
    
    # Optimization
    "Use Haiku for simple queries: 20x cheaper than Opus at similar speed.",
    "Cache embeddings to avoid re-computing for similar queries.",
    "Semantic chunking improves retrieval quality without extra cost.",
    "Reranking adds ~$0.01 per query but improves precision significantly.",
    
    # Scaling
    "At 100K queries/month with Opus: ~$5,000 monthly cost.",
    "At 100K queries/month with Haiku: ~$250 monthly cost.",
    "OpenSearch Serverless auto-scales: no cluster management needed.",
    "Cross-region profiles distribute load for better availability.",
    
    # Advanced
    "Multi-modal RAG (text + images) adds Titan Multi-modal embeddings cost.",
    "Self-RAG increases cost 2-3x due to critique iterations.",
    "Tree of Thoughts RAG: 3-5x cost for parallel path exploration.",
]

# Index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing knowledge base...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

# Expected output:
# Indexing knowledge base...
# ✓ Indexed 22 documents

✓ Created index: chain_of_thought_rag_docs
Indexing knowledge base...


Indexed 21/21 documents


✓ Indexed 21 documents
✓ Indexed 21 documents


## 5️⃣ Reasoning Step Structure

In [5]:
@dataclass
class ReasoningStep:
    """One step in the chain of thought"""
    step_num: int
    question: str  # What we're trying to figure out
    retrieval_query: str  # Query for this step
    retrieved_docs: List[str]  # Retrieved information
    reasoning: str  # Thought process
    result: str  # Intermediate conclusion
    is_final: bool  # Is this the final step?

print("✓ Reasoning step structure defined")

# Expected output:
# ✓ Reasoning step structure defined

✓ Reasoning step structure defined


## 6️⃣ Step Planning

In [6]:
def plan_reasoning_steps(query: str) -> List[str]:
    """
    Decompose query into logical reasoning steps.
    """
    planning_prompt = f"""
Break down this question into logical reasoning steps.

Question: {query}

Create a step-by-step plan to answer this question.
Each step should:
1. Ask a specific sub-question
2. Build on previous steps
3. Lead toward the final answer

Example for "What's the monthly cost of 50K RAG queries with Opus?":
Step 1: What is the pricing for Claude Opus on Bedrock?
Step 2: How many tokens does a typical RAG query use?
Step 3: Calculate total tokens for 50K queries
Step 4: Calculate total cost based on pricing

Respond in JSON:
{{
    "steps": [
        "step 1 question",
        "step 2 question",
        "step 3 question"
    ]
}}

Aim for 3-5 steps.
"""
    
    response = llm.generate(planning_prompt, temperature=0.3)
    
    # Parse JSON
    try:
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        if json_start >= 0 and json_end > json_start:
            json_str = response[json_start:json_end]
            data = json.loads(json_str)
            return data['steps'][:MAX_REASONING_STEPS]
    except Exception as e:
        print(f"  Warning: Could not parse plan: {e}")
    
    # Fallback: simple decomposition
    return [
        f"Understand what the question is asking: {query}",
        "Gather relevant information from the knowledge base",
        "Analyze the information and draw conclusions"
    ]

print("✓ Step planning function ready")

# Expected output:
# ✓ Step planning function ready

✓ Step planning function ready


## 7️⃣ Step Execution

In [7]:
def execute_reasoning_step(step_num: int,
                           step_question: str,
                           original_query: str,
                           previous_steps: List[ReasoningStep]) -> ReasoningStep:
    """
    Execute one reasoning step: retrieve → think → conclude.
    """
    print(f"\n  Step {step_num}: {step_question[:60]}...")
    
    # Build context from previous steps
    context_summary = ""
    if previous_steps:
        context_summary = "Previous findings:\n"
        for prev in previous_steps:
            context_summary += f"  - Step {prev.step_num}: {prev.result[:80]}...\n"
    
    # Retrieve for this step
    retrieval_query = f"{step_question} {original_query}"
    query_emb = embedder.embed_text(retrieval_query)
    retrieved_docs = opensearch.vector_search(query_emb, top_k=RETRIEVAL_TOP_K)
    context_docs = [doc['text'] for doc in retrieved_docs]
    
    print(f"    Retrieved: {len(context_docs)} docs")
    
    # Reason through this step
    reasoning_prompt = f"""
You are working through a reasoning chain to answer a question.

Original Question: {original_query}

{context_summary}

Current Step ({step_num}): {step_question}

Retrieved Information:
{chr(10).join([f'[{i+1}] {doc}' for i, doc in enumerate(context_docs)])}

Think through this step:
1. What information from the retrieved docs helps answer this step?
2. How does this connect to previous steps?
3. What can we conclude from this step?

Respond with:
- Your reasoning process
- The conclusion for this step
- Whether this is enough to answer the original question (true/false)

Format:
REASONING: [your thought process]
CONCLUSION: [what we learned from this step]
FINAL: [true if we can answer the original question, false if need more steps]
"""
    
    response = llm.generate(reasoning_prompt, temperature=0.7)
    
    # Parse response
    reasoning = "Processed information"
    result = "Intermediate conclusion"
    is_final = False
    
    if "REASONING:" in response:
        reasoning_start = response.find("REASONING:") + len("REASONING:")
        reasoning_end = response.find("CONCLUSION:") if "CONCLUSION:" in response else len(response)
        reasoning = response[reasoning_start:reasoning_end].strip()
    
    if "CONCLUSION:" in response:
        conclusion_start = response.find("CONCLUSION:") + len("CONCLUSION:")
        conclusion_end = response.find("FINAL:") if "FINAL:" in response else len(response)
        result = response[conclusion_start:conclusion_end].strip()
    
    if "FINAL:" in response:
        final_text = response[response.find("FINAL:") + len("FINAL:"):].strip().lower()
        is_final = "true" in final_text
    
    print(f"    Reasoning: {reasoning[:60]}...")
    print(f"    Result: {result[:60]}...")
    print(f"    Final step: {is_final}")
    
    return ReasoningStep(
        step_num=step_num,
        question=step_question,
        retrieval_query=retrieval_query,
        retrieved_docs=context_docs,
        reasoning=reasoning,
        result=result,
        is_final=is_final
    )

print("✓ Step execution function ready")

# Expected output:
# ✓ Step execution function ready

✓ Step execution function ready


## 8️⃣ Chain of Thought RAG Pipeline

In [8]:
def chain_of_thought_rag(query: str, max_steps: int = 5) -> Dict:
    """
    Chain of Thought RAG: step-by-step reasoning with retrieval.
    
    Returns:
        Dict with answer, reasoning chain, metadata
    """
    start_time = time.time()
    
    print(f"Query: {query}\n")
    print("="*70)
    
    # Step 1: Plan reasoning steps
    print("\nStep 1: Planning Reasoning Chain")
    step_questions = plan_reasoning_steps(query)
    
    print(f"  Planned {len(step_questions)} reasoning steps:")
    for i, q in enumerate(step_questions, 1):
        print(f"    {i}. {q[:70]}...")
    
    # Step 2: Execute reasoning chain
    print(f"\nStep 2: Executing Reasoning Chain")
    
    reasoning_chain = []
    
    for i, step_q in enumerate(step_questions, 1):
        step = execute_reasoning_step(
            step_num=i,
            step_question=step_q,
            original_query=query,
            previous_steps=reasoning_chain
        )
        
        reasoning_chain.append(step)
        
        # Early stopping if we have final answer
        if step.is_final:
            print(f"\n  ✓ Reached final answer at step {i}")
            break
    
    # Step 3: Synthesize final answer
    print(f"\nStep 3: Synthesizing Final Answer")
    
    # Build complete reasoning context
    reasoning_summary = "\n\n".join([
        f"Step {s.step_num}: {s.question}\n"
        f"Reasoning: {s.reasoning}\n"
        f"Conclusion: {s.result}"
        for s in reasoning_chain
    ])
    
    synthesis_prompt = f"""
You have completed a step-by-step reasoning process.

Original Question: {query}

Reasoning Chain:
{reasoning_summary}

Based on this reasoning chain, provide a clear, comprehensive final answer to the original question.
The answer should:
1. Directly address the question
2. Incorporate insights from all steps
3. Be well-structured and complete

Final Answer:
"""
    
    final_answer = llm.generate(synthesis_prompt, temperature=0.7)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print("COMPLETED")
    print(f"{'='*70}")
    print(f"  Total time: {total_time:.2f}s")
    print(f"  Reasoning steps: {len(reasoning_chain)}")
    print()
    
    return {
        'answer': final_answer,
        'reasoning_chain': reasoning_chain,
        'metadata': {
            'total_time': total_time,
            'num_steps': len(reasoning_chain),
            'total_retrievals': len(reasoning_chain) * RETRIEVAL_TOP_K,
            'planned_steps': len(step_questions),
            'early_stop': len(reasoning_chain) < len(step_questions)
        }
    }

print("✓ Chain of Thought RAG pipeline ready")

# Expected output:
# ✓ Chain of Thought RAG pipeline ready

✓ Chain of Thought RAG pipeline ready


## 9️⃣ Test Chain of Thought RAG

In [9]:
# Test with queries that benefit from step-by-step reasoning
test_queries = [
    "What's the monthly cost difference between using Opus vs Haiku for 50,000 RAG queries?",
    "If I want to optimize RAG costs while maintaining quality, what should I do?",
]

results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'#'*70}")
    print(f"# TEST {i}/{len(test_queries)}")
    print(f"{'#'*70}\n")
    
    result = chain_of_thought_rag(query, max_steps=MAX_REASONING_STEPS)
    results.append(result)
    
    print("\n" + "="*70)
    print("REASONING CHAIN")
    print("="*70)
    
    for step in result['reasoning_chain']:
        print(f"\n📍 Step {step.step_num}: {step.question}")
        print(f"   💭 Reasoning: {step.reasoning[:100]}...")
        print(f"   ✓ Result: {step.result[:100]}...")
    
    print("\n" + "="*70)
    print("FINAL ANSWER")
    print("="*70)
    print(f"\n{result['answer'][:400]}...\n")
    print(f"Steps used: {result['metadata']['num_steps']}/{result['metadata']['planned_steps']}")
    print(f"Total retrievals: {result['metadata']['total_retrievals']}")
    print(f"\n{'#'*70}\n")

# Expected output:
# [Shows step-by-step reasoning with retrieval at each step]


######################################################################
# TEST 1/2
######################################################################

Query: What's the monthly cost difference between using Opus vs Haiku for 50,000 RAG queries?


Step 1: Planning Reasoning Chain


  Planned 5 reasoning steps:
    1. What are the input and output token prices for Claude Opus on Amazon B...
    2. What are the input and output token prices for Claude Haiku on Amazon ...
    3. How many input and output tokens does a typical RAG query consume (inc...
    4. What is the total monthly cost for 50,000 RAG queries using Opus vs Ha...
    5. What is the absolute and percentage cost difference between Opus and H...

Step 2: Executing Reasoning Chain

  Step 1: What are the input and output token prices for Claude Opus o...


    Retrieved: 0 docs


    Reasoning: 1. The retrieved information for this step appears to be emp...
    Result: Based on known pricing (from training data), Claude 3 Opus o...
    Final step: False

  Step 2: What are the input and output token prices for Claude Haiku ...
    Retrieved: 0 docs


    Reasoning: The retrieved information for this step appears to be empty ...
    Result: Claude 3 Haiku on Amazon Bedrock is priced at approximately ...
    Final step: True

  ✓ Reached final answer at step 2

Step 3: Synthesizing Final Answer


...

  Planned 5 reasoning steps:
    1. What are the main cost drivers in a RAG pipeline (embedding, retrieval...
    2. What optimization strategies exist for each cost driver (e.g., smaller...
    3. What quality trade-offs are associated with each optimization strategy...
    4. Which combination of optimizations provides the best cost reduction wh...
    5. What is the recommended prioritized action plan for implementing these...

Step 2: Executing Reasoning Chain

  Step 1: What are the main cost drivers in a RAG pipeline (embedding,...
    Retrieved: 0 docs


    Reasoning: Let me think through the main cost drivers in a RAG (Retriev...
    Result: The main cost drivers in a RAG pipeline are: (1) LLM inferen...
    Final step: False

  Step 2: What optimization strategies exist for each cost driver (e.g...
    Retrieved: 0 docs


    Reasoning: Let me work through this step carefully, even though the ret...
    Result: For each cost driver in RAG, specific optimization strategie...
    Final step: False

  Step 3: What quality trade-offs are associated with each optimizatio...
    Retrieved: 3 docs


    Reasoning: 1. **What information from the retrieved docs helps answer t...
    Result: The retrieved information provides limited direct evidence o...
    Final step: True

  ✓ Reached final answer at step 3

Step 3: Synthesizing Final Answer



COMPLETED
  Total time: 68.07s
  Reasoning steps: 3


REASONING CHAIN

📍 Step 1: What are the main cost drivers in a RAG pipeline (embedding, retrieval, LLM inference, storage)?
   💭 Reasoning: Let me think through the main cost drivers in a RAG (Retrieval-Augmented Generation) pipeline, even ...
   ✓ Result: The main cost drivers in a RAG pipeline are: (1) LLM inference - typically the largest cost, driven ...

📍 Step 2: What optimization strategies exist for each cost driver (e.g., smaller embedding models, chunk size tuning, caching, model tier selection)?
   💭 Reasoning: Let me work through this step carefully, even though the retrieved information appears to be empty o...
   ✓ Result: For each cost driver in RAG, specific optimization strategies exist: (1) LLM costs can be reduced th...

📍 Step 3: What quality trade-offs are associated with each optimization strategy (e.g., retrieval accuracy loss, response degradation)?
   💭 Reasoning: 1. **What information from the retrieved do

## 🔟 Comparison: Chain of Thought vs Simple RAG

In [10]:
comparison_query = "How much would it cost to run a RAG system serving 100K queries per month?"

print("="*70)
print("CHAIN OF THOUGHT RAG (Step-by-Step)")
print("="*70 + "\n")

cot_result = chain_of_thought_rag(comparison_query, max_steps=MAX_REASONING_STEPS)

print("\n" + "="*70)
print("SIMPLE RAG (Single-Shot)")
print("="*70 + "\n")

# Simple RAG: single retrieval and generation
simple_start = time.time()
query_emb = embedder.embed_text(comparison_query)
simple_docs = opensearch.vector_search(query_emb, top_k=5)
simple_context = [d['text'] for d in simple_docs]
simple_answer = llm.generate_with_context(comparison_query, simple_context)
simple_time = time.time() - simple_start

print(f"Retrieved {len(simple_docs)} docs")
print(f"Generated answer in {simple_time:.2f}s")

# Comparison
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

print(f"\n🔗 Reasoning Process:")
print(f"  Chain of Thought: {cot_result['metadata']['num_steps']} explicit steps")
print(f"  Simple: 1 step (black box)")

print(f"\n📚 Retrieval Strategy:")
print(f"  Chain of Thought: {cot_result['metadata']['total_retrievals']} targeted retrievals")
print(f"  Simple: {len(simple_docs)} docs upfront")

print(f"\n👁️  Transparency:")
print(f"  Chain of Thought: Full reasoning visible")
for i, step in enumerate(cot_result['reasoning_chain'], 1):
    print(f"    Step {i}: {step.question[:50]}...")
print(f"  Simple: No intermediate steps")

print(f"\n⏱️  Latency:")
print(f"  Chain of Thought: {cot_result['metadata']['total_time']:.2f}s (multiple steps)")
print(f"  Simple: {simple_time:.2f}s (single pass)")

print(f"\n💰 Cost (estimated):")
cot_cost = 0.05 * (cot_result['metadata']['num_steps'] + 1)  # Each step + synthesis
simple_cost = 0.05
print(f"  Chain of Thought: ~${cot_cost:.3f} (multiple LLM calls)")
print(f"  Simple: ~${simple_cost:.3f} (baseline)")

print(f"\n🐛 Debuggability:")
print(f"  Chain of Thought: Can trace exact reasoning flow")
print(f"  Simple: Hard to debug if answer is wrong")

print(f"\n📝 Answers (First 250 chars):\n")
print(f"Chain of Thought:\n{cot_result['answer'][:250]}...\n")
print(f"Simple RAG:\n{simple_answer[:250]}...")

print(f"\n💡 Key Advantage:")
print(f"  Chain of Thought shows HOW it reached the answer,")
print(f"  making reasoning traceable and verifiable.")

# Expected output:
# [Shows Chain of Thought's transparent reasoning vs Simple RAG's black box]

CHAIN OF THOUGHT RAG (Step-by-Step)

Query: How much would it cost to run a RAG system serving 100K queries per month?


Step 1: Planning Reasoning Chain


  Planned 5 reasoning steps:
    1. What are the main cost components of a RAG system (e.g., LLM inference...
    2. What is the pricing for each component (e.g., LLM input/output token c...
    3. What is the average token usage per RAG query (context retrieval + pro...
    4. Calculate the total cost per component for 100K queries per month base...
    5. Sum all component costs to determine the total monthly cost range for ...

Step 2: Executing Reasoning Chain

  Step 1: What are the main cost components of a RAG system (e.g., LLM...
    Retrieved: 3 docs


    Reasoning: Let me analyze the retrieved information to identify the mai...
    Result: The main cost components of a RAG system include: (1) LLM in...
    Final step: False

  Step 2: What is the pricing for each component (e.g., LLM input/outp...
    Retrieved: 3 docs


    Reasoning: 1. **What information from the retrieved docs helps answer t...
    Result: Pricing breakdown for 100K queries/month:
- **LLM inference*...
    Final step: False

  Step 3: What is the average token usage per RAG query (context retri...
    Retrieved: 3 docs


    Reasoning: 1. **What information from the retrieved docs helps answer t...
    Result: A typical RAG query consumes ~700 tokens total (500 input co...
    Final step: True

  ✓ Reached final answer at step 3

Step 3: Synthesizing Final Answer



COMPLETED
  Total time: 54.84s
  Reasoning steps: 3


SIMPLE RAG (Single-Shot)



Retrieved 5 docs
Generated answer in 5.68s

COMPARISON

🔗 Reasoning Process:
  Chain of Thought: 3 explicit steps
  Simple: 1 step (black box)

📚 Retrieval Strategy:
  Chain of Thought: 9 targeted retrievals
  Simple: 5 docs upfront

👁️  Transparency:
  Chain of Thought: Full reasoning visible
    Step 1: What are the main cost components of a RAG system ...
    Step 2: What is the pricing for each component (e.g., LLM ...
    Step 3: What is the average token usage per RAG query (con...
  Simple: No intermediate steps

⏱️  Latency:
  Chain of Thought: 54.84s (multiple steps)
  Simple: 5.68s (single pass)

💰 Cost (estimated):
  Chain of Thought: ~$0.200 (multiple LLM calls)
  Simple: ~$0.050 (baseline)

🐛 Debuggability:
  Chain of Thought: Can trace exact reasoning flow
  Simple: Hard to debug if answer is wrong

📝 Answers (First 250 chars):

Chain of Thought:
# Cost Estimate: RAG System at 100K Queries/Month

## Summary

Running a RAG system at 100K queries/month typically costs **$20

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Step-by-step reasoning decomposition
✅ Targeted retrieval at each step
✅ Transparent intermediate reasoning
✅ Progressive answer building
✅ Traceable logic flow

### Performance Characteristics

- **Latency**: 3-4x Simple RAG (multiple steps)
- **Cost**: 2-4x Simple RAG (multiple LLM calls)
- **Transparency**: Full reasoning visible
- **Debuggability**: Easy to trace logic
- **Quality**: Better for complex reasoning

### Chain of Thought vs Other Patterns

| Aspect | Simple RAG | Self-RAG | CoT RAG |
|--------|-----------|----------|----------|
| Reasoning | Single-shot | Iterative | Step-by-step |
| Transparency | Black box | Scores | Full steps |
| Retrieval | Once | Adaptive | Per step |
| Focus | Speed | Quality | Explainability |
| Latency | ~2s | ~5-10s | ~6-12s |
| Cost | $0.05 | $0.10-0.15 | $0.10-0.20 |
| Best for | Simple Q&A | Critical accuracy | Complex reasoning |

### When to Use Chain of Thought

**Use CoT RAG when:**
- Complex multi-step reasoning needed
- Need to explain HOW answer was reached
- Debugging poor answers
- Math or logic problems
- Educational applications
- Require verifiable reasoning

**Skip CoT when:**
- Simple factual queries
- Reasoning steps not valuable
- Tight latency budget
- Answer quality > explainability
- Cost is primary concern

### Key Insights

1. **Transparency**: Shows complete reasoning process
2. **Targeted Retrieval**: Gets info as needed per step
3. **Progressive Building**: Each step builds on previous
4. **Debuggability**: Easy to identify where reasoning fails
5. **Educational**: Great for showing thought process

### Best Practices

1. **3-5 Steps**: Most queries need 3-5 reasoning steps

2. **Clear Step Questions**: Each step should ask something specific

3. **Small Retrievals**: Top-3 per step > Top-10 once

4. **Early Stopping**: Stop when answer is ready

5. **Context Propagation**: Pass previous steps to current

6. **Synthesis**: Final step should synthesize all findings

7. **Log Reasoning**: Keep full chain for debugging

8. **Temperature 0.3-0.7**: Lower for planning, higher for reasoning

### Advanced Techniques

- **Adaptive Steps**: Generate more steps if needed
- **Parallel Branches**: Explore multiple reasoning paths
- **Step Validation**: Verify each step before proceeding
- **Sub-step Expansion**: Break complex steps into sub-steps
- **Backtracking**: Retry if step fails
- **Interactive**: Ask user for clarification at steps

### Limitations

1. **Higher Latency**: Multiple LLM calls sequential
2. **Higher Cost**: Each step costs LLM call
3. **Over-engineering**: Overkill for simple queries
4. **Step Quality**: Bad plan → bad answer
5. **No Parallelism**: Steps must run sequentially

### Real-world Applications

**Where CoT RAG Excels:**
- Math word problems (step-by-step calculation)
- Financial analysis (break down calculations)
- Legal reasoning (trace argument logic)
- Medical diagnosis (differential diagnosis steps)
- Educational tutoring (show work process)

### Cost Breakdown (per query)

**4-step example:**
- Planning: $0.02
- Step 1 (retrieve + reason): $0.03
- Step 2 (retrieve + reason): $0.03
- Step 3 (retrieve + reason): $0.03
- Step 4 (retrieve + reason): $0.03
- Synthesis: $0.04
- **Total**: ~$0.18 (vs $0.05 Simple RAG)

**Worth it?** When explainability matters: 3.6x cost → full reasoning trace

### Comparison with Related Patterns

**vs Self-RAG:**
- Self-RAG: Critiques and refines (quality focus)
- CoT: Explains reasoning steps (transparency focus)

**vs Tree of Thoughts:**
- Tree: Explores multiple paths in parallel
- CoT: Follows single path step-by-step

**Combined:** CoT + Self-RAG = step-by-step with quality checks

### Transparency Benefits

Visible reasoning helps:
- **Debugging**: See where logic fails
- **Trust**: Verify reasoning is sound
- **Learning**: Understand how to solve problems
- **Auditing**: Trace decision-making process
- **Improvement**: Identify weak steps

### Next Steps

- **Validate Steps**: Add verification per step
- **Optimize Planning**: Better step decomposition
- **Parallel Sub-steps**: When steps are independent
- **User Interaction**: Allow mid-chain clarification
- **Step Templates**: Pre-defined reasoning patterns

---

## 🎉 Phase 2 Progress!

**Progress**: 16/37 patterns (43%)
- Phase 1: 10/10 ✅ Complete
- Phase 2: 6/12 ✅ Multi-modal + Agentic + CRAG + Self-RAG + Tree of Thoughts + Chain of Thought

**Next**: ReAct RAG - Reason + Act cycles with interleaved thinking

## 🧹 Cleanup

In [11]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")

# Expected output:
# 
# To delete the index later:
#   opensearch.delete_index('chain_of_thought_rag_docs')


To delete the index later:
  opensearch.delete_index('chain_of_thought_rag_docs')
